In [0]:
dbutils.widgets.text("p_batch_id", "")

In [0]:
v_batch_id = dbutils.widgets.get("p_batch_id")
print(v_batch_id)

In [0]:
%run ../00-common/01_Environmnet_config

In [0]:
%run ../00-common/02_bronze_helpers

In [0]:
silver_table = f"{catalog_name}.{silver_schema}.constructors"
gold_nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"
dim_constructors = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
constructor_df = spark.read.table(silver_table).filter(F.col("batch_id") == v_batch_id)
gold_nationality_df = spark.read.table(gold_nationality_table)

In [0]:
dim_constructors_df = constructor_df.join(
    gold_nationality_df,
    constructor_df.nationality == gold_nationality_df.nationality,
    "left",
).select(
    constructor_df.constructor_id,
    constructor_df.constructor_name,
    constructor_df.nationality,
    gold_nationality_df.region.alias("nationality_region"),
)

In [0]:
write_to_gold(
    dim_constructors_df,
    dim_constructors,
    "t.constructor_id = s.constructor_id",
     columns_to_update = [
        "constructors_name",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
%sql
select
  *
from
  formula1_incr.gold.dim_constructors